# Lab 02: first steps with llama cli

Lab: `corso/laboratori/lab-02-01-primo-modello, lab-02-02-campionamento, lab-02-03-chat-template`

Run the cells in order (Runtime > Run all). It downloads up to 1 GB and takes a few minutes.


## 1. Setup

Download the official llama.cpp binaries (build `b11046`, the one used in the course) and the lab files.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # threads = available cores (more than the physical cores slows things down)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Download the sample model (785 MiB, Apache 2.0) into the Hugging Face cache.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. First question (lab 02-01)

`-st` answers and exits, `-rea off` turns reasoning off, `-t` sets the threads.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Quanto fa 17 per 3? Solo il numero." -n 30 -rea off -t $T < /dev/null 2>&1 | tail -8

With reasoning on and few tokens the model stays inside `[Start thinking]` without answering; `--reasoning-budget 25` cuts the thinking short so the answer arrives.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Quanto fa 17 per 3?" -n 80 -t $T --reasoning-budget 25 < /dev/null 2>&1 | tail -8

### System prompt

Same question, now with context.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -rea off -t $T -n 60 \
  -sys "Sei un esperto di modelli linguistici (LLM). Rispondi in italiano, in una frase." \
  -p "Cos'è un token?" < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'

## 3. Sampling (lab 02-02)

A fixed seed makes the run repeatable; `--temp 0` and `--top-k 1` always pick the most likely token.

In [ ]:
%%bash
P="Scrivi una frase su un gatto."
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "$P" -n 25 -rea off -t $T --temp 0 < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "$P" -n 25 -rea off -t $T -s 1 --temp 1.8 --top-k 0 --top-p 1 --min-p 0.1 < /dev/null 2>&1 | sed -n '/^> /,/Prompt:/p'

## 4. Chat template (lab 02-03)

The verbose log shows the format `llama` uses to lay out messages.

In [ ]:
%%bash
timeout 300 llama cli -hf ggml-org/Qwen3.5-0.8B-GGUF -st -p "Ciao" -n 1 -rea off -t $T -v < /dev/null 2>&1 | grep -A6 "example_format"